# اليوم الأول — الانتباه والمحولات
## Day 1 — Attention & Transformers

**المدربة / Instructor:** ميعاد المري — Meaad Al-Marri  
**المسار / Track:** Core → Explore → Distinction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/almiyead-rgb/bayan-applied-nlp-course/blob/develop/notebooks/02_attention_transformers.ipynb)

> السؤال المحوري: كيف يقرر كل Token أي Tokens أخرى يحتاجها لبناء تمثيل سياقي؟
>
> Driving question: How does each token decide which other tokens it needs for a contextual representation?

**النواتج / Outcomes:** تنفيذ Scaled Dot-Product Attention، قراءة الأشكال، تطبيق القناع، فهم Multi-Head Attention وEncoder Block، وتفسير حدود مصفوفة الانتباه.

## 0) خريطة المفاهيم / Concept map

- **Query (Q):** ما الذي أبحث عنه؟
- **Key (K):** ما الصفات التي أعرضها للمطابقة؟
- **Value (V):** ما المعلومة التي سأمررها إذا حصلت على وزن مرتفع؟

\[
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}+M\right)V
\]

سننفذها بـNumPy كي نرى كل خطوة. PyTorch اختياري للمقارنة.

**المتوقع / Expected:** في النهاية تظهر `DAY1_NOTEBOOK2_CORE=PASS`.

In [ ]:
import math
import numpy as np

SEED = 42
rng = np.random.default_rng(SEED)
print("NumPy:", np.__version__)
print("Core runtime ready / بيئة Core جاهزة")

## 1) تنفيذ Scaled Dot-Product Attention

| Tensor | Shape | المعنى |
|---|---:|---|
| Q | `(T_q, d_k)` | استعلام لكل موضع |
| K | `(T_k, d_k)` | مفتاح لكل موضع |
| V | `(T_k, d_v)` | قيمة لكل موضع |
| Scores | `(T_q, T_k)` | درجة كل استعلام مع كل مفتاح |
| Output | `(T_q, d_v)` | التمثيل السياقي الجديد |

In [ ]:
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(shifted)
    return exp_x / exp_x.sum(axis=axis, keepdims=True)


def scaled_dot_product_attention(
    q: np.ndarray,
    k: np.ndarray,
    v: np.ndarray,
    keep_mask: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if q.shape[-1] != k.shape[-1]:
        raise ValueError("Q and K must share d_k")
    if k.shape[-2] != v.shape[-2]:
        raise ValueError("K and V must share sequence length")
    scores = q @ np.swapaxes(k, -1, -2) / math.sqrt(q.shape[-1])
    if keep_mask is not None:
        mask = np.asarray(keep_mask, dtype=bool)
        try:
            mask = np.broadcast_to(mask, scores.shape)
        except ValueError as exc:
            raise ValueError("keep_mask is not broadcastable to scores") from exc
        if np.any(mask.sum(axis=-1) == 0):
            raise ValueError("Every query row must keep at least one key")
        scores = np.where(mask, scores, -np.inf)
    weights = softmax(scores, axis=-1)
    output = weights @ v
    return output, weights, scores

q = np.array([[1.0, 0.0], [0.0, 1.0]])
k = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
v = np.array([[10.0, 0.0], [0.0, 10.0], [5.0, 5.0]])
output, weights, scores = scaled_dot_product_attention(q, k, v)
print("scores shape:", scores.shape)
print("weights:\n", np.round(weights, 3))
print("row sums:", weights.sum(axis=-1))
print("output:\n", np.round(output, 3))
assert output.shape == (2, 2)
assert np.allclose(weights.sum(axis=-1), 1.0)
print("Scaled attention=PASS")

**المتوقع / Expected:** Scores بحجم `(2, 3)`، ومجموع كل صف من Weights يساوي 1، ثم رسالة نجاح.

### لماذا القسمة على \(\sqrt{d_k}\)؟

عندما يكبر البعد قد تكبر الضربات النقطية، فتدخل Softmax مناطق شديدة الحدة وتضعف التدرجات. التحجيم يحافظ عادة على قيم أكثر استقرارًا.

In [ ]:
def entropy(probabilities: np.ndarray) -> float:
    p = np.clip(probabilities, 1e-12, 1.0)
    return float(-np.sum(p * np.log(p), axis=-1).mean())

q_big = rng.normal(size=(6, 64))
k_big = rng.normal(size=(6, 64))
raw_scores = q_big @ k_big.T
unscaled_weights = softmax(raw_scores)
scaled_weights = softmax(raw_scores / math.sqrt(q_big.shape[-1]))
print("mean entropy without scaling:", round(entropy(unscaled_weights), 3))
print("mean entropy with scaling:   ", round(entropy(scaled_weights), 3))
print("Scaling comparison complete")

## 2) الأقنعة / Masks

في هذه الحزمة نستخدم عقدًا واضحًا: `True` يعني **يسمح بالمشاركة** و`False` يعني **يُحجب**. قد تختلف الدلالة بين المكتبات والواجهات؛ اقرأ توثيق الدالة دائمًا.

القناع السببي يمنع الموضع من رؤية المستقبل. Encoder مثل BERT يستخدم عادة Padding Mask، وليس Causal Mask في الضبط المعتاد.

In [ ]:
causal_keep_mask = np.tril(np.ones((2, 3), dtype=bool))
masked_output, masked_weights, _ = scaled_dot_product_attention(q, k, v, keep_mask=causal_keep_mask)
print("keep mask:\n", causal_keep_mask.astype(int))
print("masked weights:\n", np.round(masked_weights, 3))
assert masked_weights[0, 1] == 0.0
assert masked_weights[0, 2] == 0.0
print("Mask semantics=PASS")

## 3) من رأس واحد إلى Multi-Head

تقسم الرؤوس البعد `d_model` إلى أجزاء. كل رأس يتعلم إسقاطات مختلفة لـQ/K/V، ثم تدمج المخرجات. تعدد الرؤوس يتيح علاقات مختلفة، لكنه لا يضمن أن كل رأس يملك تفسيرًا لغويًا بشريًا.

In [ ]:
def split_heads(x: np.ndarray, num_heads: int) -> np.ndarray:
    batch, seq_len, d_model = x.shape
    if d_model % num_heads != 0:
        raise ValueError("d_model must be divisible by num_heads")
    d_head = d_model // num_heads
    return x.reshape(batch, seq_len, num_heads, d_head).transpose(0, 2, 1, 3)


def combine_heads(x: np.ndarray) -> np.ndarray:
    batch, num_heads, seq_len, d_head = x.shape
    return x.transpose(0, 2, 1, 3).reshape(batch, seq_len, num_heads * d_head)

x = rng.normal(size=(2, 5, 12))
heads = split_heads(x, num_heads=3)
combined = combine_heads(heads)
print("input:", x.shape)
print("split:", heads.shape, "= (batch, heads, tokens, d_head)")
print("combined:", combined.shape)
assert heads.shape == (2, 3, 5, 4)
assert np.allclose(combined, x)
print("Multi-head shape journey=PASS")

## 4) Encoder Block وBERT

Encoder block مبسط:

1. Multi-Head Self-Attention.
2. Residual connection + Layer Normalization.
3. Feed-Forward Network لكل موضع.
4. Residual connection + Layer Normalization.

تضاف معلومات الموضع لأن الانتباه وحده لا يعرف ترتيب الكلمات. BERT يبني مكدس Encoder ثنائي السياق؛ اختيار `[CLS]` للتصنيف أو تمثيل كل Token لـNER يعتمد على المهمة.

In [ ]:
try:
    import torch
    torch.manual_seed(SEED)
    layer = torch.nn.TransformerEncoderLayer(
        d_model=12, nhead=3, dim_feedforward=24,
        dropout=0.0, batch_first=True, norm_first=False,
    )
    layer.eval()
    torch_x = torch.tensor(x, dtype=torch.float32)
    with torch.no_grad():
        encoder_output = layer(torch_x)
    print("PyTorch encoder output:", tuple(encoder_output.shape))
    assert tuple(encoder_output.shape) == (2, 5, 12)
    print("Optional encoder layer=PASS")
except ModuleNotFoundError:
    print("PyTorch is optional here; NumPy Core remains complete.")

## 5) مقارنة اختيارية مع PyTorch SDPA

إذا كانت PyTorch متاحة، نقارن تنفيذ NumPy مع الدالة الرسمية بأبعاد صغيرة وبدون Dropout.

In [ ]:
try:
    import torch
    import torch.nn.functional as F
    tq = torch.tensor(q, dtype=torch.float64)[None, None, :, :]
    tk = torch.tensor(k, dtype=torch.float64)[None, None, :, :]
    tv = torch.tensor(v, dtype=torch.float64)[None, None, :, :]
    torch_result = F.scaled_dot_product_attention(tq, tk, tv, dropout_p=0.0)
    np_result = output[None, None, :, :]
    max_difference = float(np.max(np.abs(torch_result.numpy() - np_result)))
    print("max NumPy/PyTorch difference:", f"{max_difference:.3e}")
    assert max_difference < 1e-9
    print("NumPy/PyTorch parity=PASS")
except ModuleNotFoundError:
    print("PyTorch unavailable; parity check skipped without affecting Core.")

## 6) ما الذي لا تثبته خريطة الانتباه؟

قد تساعد الأوزان على الفحص، لكنها لا تثبت وحدها سبب قرار النموذج ولا تساوي تفسيرًا سببيًا. افصل بين:

- **وصف:** أين وضعت طبقة/رأس أوزانًا أعلى؟
- **ادعاء تفسيري:** لماذا اتخذ النموذج القرار؟ يحتاج اختبارات إضافية كالإزالة أو التبديل أو أساليب تفسير أخرى.

لا تعرض Heatmap واحدة كبرهان نهائي.

## 7) تمارين المستويات / Challenge lanes

- **Core:** غيّر V واشرح كيف يتغير Output مع ثبات الأوزان.
- **Explore:** أنشئ Padding Mask لدفعة من تسلسلين وتحقق أن الحشو يحصل على وزن صفر.
- **Distinction:** قارن زمن التنفيذ لأطوال `T=32,64,128,256` واشرح لماذا ذاكرة Scores تنمو تقريبًا مع \(T^2\). لا تعمم نتيجة جهاز واحد على كل البيئات.

### سؤال قرار / Decision prompt

اكتب في 3–5 أسطر: أي قناع تحتاجه في مشروعك؟ ما شكل Q/K/V المتوقع؟ وما الخطأ الذي سيظهر لو انعكست دلالة القناع؟

In [ ]:
core_checks = {
    "weights_sum_to_one": np.allclose(weights.sum(axis=-1), 1.0),
    "output_shape": output.shape == (2, 2),
    "masked_positions_zero": masked_weights[0, 1] == 0 and masked_weights[0, 2] == 0,
    "heads_round_trip": np.allclose(combined, x),
}
for name, passed in core_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")
assert all(core_checks.values())
print("DAY1_NOTEBOOK2_CORE=PASS")

## نقطة GitHub / GitHub checkpoint

1. احفظ نسخة في مستودعك العام داخل `notebooks/`.
2. Commit: `feat(day1): implement scaled attention lab`.
3. أضف إلى تقرير اليوم جدول Shapes وشرح القناع، لا مجرد مخرجات منسوخة.
4. اربط Commit في `README.md`.

**دليل الاكتمال:** `DAY1_NOTEBOOK2_CORE=PASS` + إجابة سؤال القرار + رابط Commit عام.

**التالي:** ارجع إلى [صفحة مختبرات اليوم الأول](../day-01/04-labs-checkpoint.md) لإكمال بوابة المشروع Gate A.